<a href="https://colab.research.google.com/github/phaneendra520/MSAI-Develop/blob/main/TraditionalChatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install nltk scikit-learn gradio
import nltk, json, random, re, numpy as np
nltk.download('punkt'); nltk.download('punkt_tab'); nltk.download('wordnet'); nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [2]:
{"intents": [
  {"tag": "greeting", "patterns": ["hi", "hello", "good morning", "hey there"],
   "responses": ["Hello! How can I help you today?"]},
  {"tag": "password_reset", "patterns": ["I forgot my password", "reset my password", "can't log in"],
   "responses": ["You can reset your password at the self-service portal."]}
]}

{'intents': [{'tag': 'greeting',
   'patterns': ['hi', 'hello', 'good morning', 'hey there'],
   'responses': ['Hello! How can I help you today?']},
  {'tag': 'password_reset',
   'patterns': ['I forgot my password', 'reset my password', "can't log in"],
   'responses': ['You can reset your password at the self-service portal.']}]}

In [4]:
%%writefile intents.json
{"intents": [
  {"tag": "greeting", "patterns": ["hi", "hello", "hey", "good morning", "good evening", "hey there"],
   "responses": ["Hello! How can I help you today?", "Hi there! What can I do for you?"]},
  {"tag": "goodbye", "patterns": ["bye", "goodbye", "see you later", "talk to you later", "I am leaving"],
   "responses": ["Goodbye! Have a great day.", "See you later!"]},
  {"tag": "thanks", "patterns": ["thanks", "thank you", "that helps", "appreciate it", "thanks a lot"],
   "responses": ["You're welcome!", "Happy to help!"]},
  {"tag": "password_reset", "patterns": ["I forgot my password", "reset my password", "cannot log in", "locked out of my account", "change password"],
   "responses": ["You can reset your password at the self-service portal, or call the help desk."]},
  {"tag": "wifi_help", "patterns": ["wifi is not working", "cannot connect to wifi", "internet is slow", "how do I connect to the network", "no internet"],
   "responses": ["Connect to the campus network with your student ID and password. If it still fails, forget the network and reconnect."]},
  {"tag": "library_hours", "patterns": ["when is the library open", "library hours", "is the library open on weekends", "what time does the library close"],
   "responses": ["The library is open 8 AM to 10 PM on weekdays and 10 AM to 6 PM on weekends."]},
  {"tag": "contact_support", "patterns": ["how do I contact support", "help desk phone number", "talk to a human", "email support", "I need an agent"],
   "responses": ["You can reach the help desk at helpdesk@example.edu or 555-0100."]}
]}

Writing intents.json


In [5]:

import nltk, json, random, re

for pkg in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'stopwords']:
    nltk.download(pkg, quiet=True)

from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

lem = WordNetLemmatizer()

def clean(text):
    text = re.sub(r'[^a-z\s]', '', text.lower())
    return ' '.join(lem.lemmatize(w) for w in nltk.word_tokenize(text))

with open('intents.json') as f:
    data = json.load(f)

X, y = [], []
for it in data['intents']:
    for p in it['patterns']:
        X.append(clean(p))
        y.append(it['tag'])

vec = TfidfVectorizer(ngram_range=(1, 2))
clf = LogisticRegression(max_iter=1000)
clf.fit(vec.fit_transform(X), y)

resp = {it['tag']: it['responses'] for it in data['intents']}

def reply(msg, threshold=0.25):
    probs = clf.predict_proba(vec.transform([clean(msg)]))[0]
    if probs.max() < threshold:
        return "Sorry, I didn't understand. Could you rephrase?"
    return random.choice(resp[clf.classes_[probs.argmax()]])

# Quick test
for q in ["hello", "I forgot my password", "wifi not working", "library hours?", "what is the weather"]:
    print(f"You: {q}\nBot: {reply(q)}\n")

You: hello
Bot: Hi there! What can I do for you?

You: I forgot my password
Bot: You can reset your password at the self-service portal, or call the help desk.

You: wifi not working
Bot: Connect to the campus network with your student ID and password. If it still fails, forget the network and reconnect.

You: library hours?
Bot: The library is open 8 AM to 10 PM on weekdays and 10 AM to 6 PM on weekends.

You: what is the weather
Bot: Sorry, I didn't understand. Could you rephrase?



In [6]:
import gradio as gr
gr.ChatInterface(lambda m, h: reply(m)).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://03ed1d497d3314ed14.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
